# 03b -- WalterFootball Rankings

**Purpose**: Scrape WalterFootball position-specific draft prospect rankings for the
class in `CFG.rookie_class_year` and ingest into `fact_rookie_rankings`. Self-contained.

**Class and phase**: the class year and the phase dates live in `LeagueConfig`
(`rookie_class_year`, `rookie_class_combine_end`, `rookie_class_draft_end`). The phase is
set by the run date through `CFG.rookie_phase(TODAY)`: `pre_combine` until the combine
ends, `post_combine` until the NFL draft ends, `post_draft` after it. `CFG.draft_year`
is not used here.

**Sources scraped**: 13 position pages (QB, RB, WR, TE, DE, DT, OLB3-4, DE3-4, NT, OLB, ILB, CB, S)

**Later classes**: a page also lists prospects of later classes, marked `*` (next class) and
`**` (the one after). They are left out, and `positional_rank` counts only the class's own
prospects.

**Aggregation**: After scraping, rows are grouped by `(player, scraped position_group)` and
`positional_rank` is averaged. A defender ranked on multiple scheme-specific pages within the
same position group (e.g., DT + NT both → DL) produces **one** ingested row with the averaged rank.
A player on pages from *different* groups (e.g., DE → DL, OLB34 → LB) produces **two** rows.

**Ingested source_names**: `WalterFootball_{position_group}` — e.g., `WalterFootball_DL`,
`WalterFootball_LB`, `WalterFootball_DB`, `WalterFootball_RB`, etc.

**`global_rank`**: null — WalterFootball does not publish a cross-position big board.

**`rank_date`**: extracted from "This page was last updated Mon. DD, YYYY" in page HTML.
Each run stores its own `rank_date` with its ranks.

**Prerequisites**: `02c_fact_rookie_rankings_seed.ipynb`, `01a_dim_rookie_prospect.ipynb`

**Outputs**: `data/fact_rookie_rankings.parquet` (appended), `data/review_fuzzy_matches.csv` (if needed)

## 1 -- Setup & Shared Helpers

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import requests
import re
import json
import time
from pathlib import Path
from datetime import date, datetime
from dataclasses import dataclass, field
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from thefuzz import fuzz, process

# ---- Shared helpers + config from etl_helpers -------------------------------
import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (
    LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS, DEFAULT_HEADERS, ETL_USER_AGENT,
    _make_session, _parse_rank_date, clean_player_name, generate_player_key,
    add_players_from_source, ingest_ranking_source, append_review,
)

In [2]:
# clean_player_name / generate_player_key / _parse_rank_date imported from etl_helpers

Helpers OK


## 2 -- add_players_from_source

In [ ]:
# add_players_from_source imported from etl_helpers (canonical, alias-aware)

## 3 -- ingest_ranking_source

In [ ]:
# ingest_ranking_source imported from etl_helpers

## 4 -- WalterFootballScraper

Players are in `<b>` tags inside `<article>`, in ranked order (positional rank =
position in document). Format: `"Name*, POS, SchoolHeight: X-X. Weight: XXX."`

In [ ]:
CLASS_YEAR = CFG.rookie_class_year

_WF_PAGES = {
    "WalterFootball_QB":    ("QBRankings",     "Quarterbacks"),
    "WalterFootball_RB":    ("RBRankings",     "Running Backs"),
    "WalterFootball_WR":    ("WRRankings",     "Wide Receivers"),
    "WalterFootball_TE":    ("TERankings",     "Tight Ends"),
    "WalterFootball_DE":    ("DERankings",     "Defensive Ends"),
    "WalterFootball_DT":    ("DTRankings",     "Defensive Tackles"),
    "WalterFootball_OLB34": ("OLB3-4Rankings", "3-4 Rush Linebackers"),
    "WalterFootball_DE34":  ("DE3-4Rankings",  "3-4 Defensive Ends"),
    "WalterFootball_NT":    ("NTRankings",     "Nose Tackles"),
    "WalterFootball_OLB":   ("OLBRankings",    "Outside Linebackers"),
    "WalterFootball_ILB":   ("ILBRankings",    "Inside Linebackers"),
    "WalterFootball_CB":    ("CBRankings",     "Cornerbacks"),
    "WalterFootball_S":     ("SRankings",      "Safeties"),
}


class WalterFootballScraper:
    # Scrapes the position-specific draft prospect rankings of CFG.rookie_class_year.
    # Each entry is a <b> inside the first <article>, in ranked order. Text format:
    # "Name*, POS, SchoolHeight: X-X. Weight: XXX. Projected Round (YYYY): X-X."
    # A trailing * marks a prospect of the next class, ** of the one after; those
    # entries are left out, and positional_rank counts only the class's own prospects.
    # An entry is a <b> that holds a "Height:" line; a name is never matched against a word list.
    # Every entry has a pick-number span; a page whose entries read fewer than its
    # spans raises, because a skipped entry would shift every later rank.

    _HEADERS = {
        "User-Agent": ETL_USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    _PLAYER_PAT = re.compile(
        r"^([A-Za-z][A-Za-z\s'‘’\-.]+?)(\*{0,2})"
        r",\s+([A-Z0-9\-/]{1,8})"
        r",\s+(.+?)(?=Height:|$)",
        re.DOTALL,
    )
    _ROUND_PAT   = re.compile(r"Projected Round \(\d{4}\):\s*([\d\-]+)")
    _UPDATED_PAT = re.compile(r"last updated\s+([A-Za-z]+\.?\s+\d{1,2},\s*\d{4})", re.IGNORECASE)
    _POS_NORM  = {"3-4DE": "DE", "3-4OLB": "OLB", "4-3DE": "DE"}

    SOURCES = {
        key: {
            "url": f"https://walterfootball.com/{stem}{CLASS_YEAR}.php",
            "site_reference": f"{CLASS_YEAR} NFL Draft Prospect Rankings: {label}",
            "source_site": "WalterFootball",
        }
        for key, (stem, label) in _WF_PAGES.items()
    }

    def __init__(self, timeout: int = 30, delay: float = 5.0):
        self.timeout = timeout
        self.delay   = delay
        self.session = _make_session(timeout_sec=timeout)
        self.session.headers.update(self._HEADERS)

    def parse(self, html: str, source_key: str):
        from bs4 import BeautifulSoup
        cfg     = self.SOURCES[source_key]
        soup    = BeautifulSoup(html, "html.parser")
        article = soup.find("article")
        if not article:
            raise ValueError(f"No <article> at {cfg['url']}")

        m_date = self._UPDATED_PAT.search(soup.get_text())
        rank_date = _parse_rank_date(m_date.group(1) if m_date else None)

        rows, read, other_class = [], 0, 0
        for b_tag in article.find_all("b"):
            text = b_tag.get_text(separator="", strip=True)
            if "Height:" not in text:
                continue
            m = self._PLAYER_PAT.match(text)
            if not m:
                print(f"  WARN {source_key}: entry not read: {text[:60]!r}")
                continue
            read += 1
            name = m.group(1).strip()
            if CLASS_YEAR + len(m.group(2)) != CLASS_YEAR:
                other_class += 1
                continue
            pos    = self._POS_NORM.get(m.group(3).strip().upper(), m.group(3).strip().upper())
            school = m.group(4).strip()
            rnd_m  = self._ROUND_PAT.search(text)
            rows.append({
                "player_name":     name,
                "position_raw":    pos,
                "school_raw":      school,
                "global_rank":     None,
                "positional_rank": len(rows) + 1,
                "grade":           None,
                "proj_round":      rnd_m.group(1) if rnd_m else None,
            })

        picks = len(article.find_all("span", class_="pick-number"))
        if read != picks:
            raise ValueError(f"{cfg['url']}: read {read} entries but the page numbers {picks}")
        if not rows:
            raise ValueError(f"No player entries parsed from {cfg['url']}")

        df = pd.DataFrame(rows)
        print(f"Scraped {source_key}: {len(df)} players ({other_class} of later classes left out) | "
              f"{cfg['site_reference']} | rank_date={rank_date}")
        return df, rank_date

    def scrape(self, source_key: str):
        cfg = self.SOURCES[source_key]
        time.sleep(self.delay)
        resp = self.session.get(cfg["url"], timeout=self.timeout)
        resp.raise_for_status()
        return self.parse(resp.text, source_key)

## 5 -- Run WalterFootball Ingestion

In [ ]:
from bs4 import BeautifulSoup   # pip install beautifulsoup4

WF_SCRAPER   = WalterFootballScraper(timeout=30)
PHASE        = CFG.rookie_phase(TODAY)
_failed      = []
_all_reviews = []
_scraped     = {}  # source_key -> (df, rank_date)
print(f"Class {CLASS_YEAR} | phase {PHASE} | run date {TODAY}")

# ── Step 1: Scrape all 13 position pages ─────────────────────────────────────
for source_key in WalterFootballScraper.SOURCES:
    print("\n" + "=" * 60)
    print(f"Scraping: {source_key}")
    try:
        rankings_df, rank_date = WF_SCRAPER.scrape(source_key)
        _scraped[source_key] = (rankings_df, rank_date)
    except Exception as e:
        print(f"  ERROR: {e}")
        _failed.append(source_key)

# ── Step 2: add_players_from_source per page (updates dim_rookie_prospect) ────
print("\n" + "=" * 60)
print("Fuzzy-matching players into dim_rookie_prospect...")
for source_key, (rankings_df, _) in _scraped.items():
    try:
        _dim_rp, review = add_players_from_source(
            rankings_df, source_name=source_key, draft_year=CLASS_YEAR
        )
        if not review.empty:
            print(f"  -> {len(review)} players flagged for review ({source_key})")
            _all_reviews.append(review)
    except Exception as e:
        print(f"  ERROR add_players ({source_key}): {e}")
        _failed.append(source_key)

# ── Step 3: Aggregate by scraped position_group (avg positional_rank) ─────────
# Players ranked on multiple pages within the same position_group get their
# positional_rank averaged — one row per player per group, no arbitrary winner.
# Players spanning different groups (e.g., DE->DL and OLB->LB) keep both rows.
print("\n" + "=" * 60)
print("Aggregating by position_group...")

_all_dfs = []
for source_key, (df, rank_date) in _scraped.items():
    _df = df.copy()
    _df["_wf_source"]   = source_key
    _df["_rank_date"]  = rank_date
    _all_dfs.append(_df)

combined_wf = pd.concat(_all_dfs, ignore_index=True)

# Join scraped position_raw -> position_group from dim_position
pos_map = etl.read_table("dim_position")[["position_raw", "position_group"]]
combined_wf = combined_wf.merge(pos_map, on="position_raw", how="left")
combined_wf["_name_clean"] = combined_wf["player_name"].apply(clean_player_name)

agg_wf = (
    combined_wf.groupby(["_name_clean", "position_group"], dropna=False)
    .agg(
        player_name    = ("player_name",    "first"),
        position_raw   = ("position_raw",   "first"),
        school_raw     = ("school_raw",     "first"),
        global_rank    = ("global_rank",    "first"),   # always null for WF
        positional_rank= ("positional_rank","mean"),
        grade          = ("grade",          "first"),   # always null for WF
        rank_date      = ("_rank_date",     "max"),     # latest date across contributing pages
        _sources       = ("_wf_source",    lambda x: "+".join(sorted(set(x)))),
    )
    .reset_index()
)
agg_wf["positional_rank"] = agg_wf["positional_rank"].round().astype("Int64")

pre  = len(combined_wf)
post = len(agg_wf)
print(f"  {pre} scraped rows -> {post} aggregated rows ({pre - post} merged by avg)")

averaged = agg_wf[agg_wf["_sources"].str.contains(r"\+", regex=True)]
if len(averaged):
    print(f"  {len(averaged)} players averaged across pages within same position_group:")
    for _, r in averaged.iterrows():
        print(f"    {r['player_name']:30s}  {r['position_group']}  "
              f"<- {r['_sources']}  (avg rank={r['positional_rank']})")

# ── Step 4: Ingest per position_group ─────────────────────────────────────────
print("\n" + "=" * 60)
print("Ingesting aggregated WalterFootball rows...")
SOURCES = {}  # populated here; used by validation cell

_nan_rows = agg_wf[agg_wf["position_group"].isna()]
if len(_nan_rows):
    print(f"WARN: {len(_nan_rows)} rows have no position_group mapping (position_raw not in dim_position) -- skipped:")
    for _, _r in _nan_rows.iterrows():
        print(f"  {_r['player_name']}  ({_r['position_raw']})")

for pos_group, group_df in agg_wf.groupby("position_group", dropna=False):
    if pd.isna(pos_group):
        continue  # already reported in WARN block above
    source_key = f"WalterFootball_{pos_group}"
    rd         = group_df["rank_date"].max()
    SOURCES[source_key] = {"source_site": "WalterFootball"}
    try:
        ingest_ranking_source(
            group_df.reset_index(drop=True),
            source_name=source_key,
            source_site="WalterFootball",
            phase=PHASE,
            draft_year=CLASS_YEAR,
            rank_date=rd,
        )
    except Exception as e:
        print(f"  ERROR ingesting {source_key}: {e}")
        _failed.append(source_key)

# ── Write review file (append-safe across notebooks) ─────────────────────────
if _all_reviews:
    new_review = pd.concat(_all_reviews, ignore_index=True)
    rp = DATA / "review" / "review_fuzzy_matches.csv"
    if rp.exists():
        existing = pd.read_csv(rp)
        combined = pd.concat([existing, new_review], ignore_index=True)
        # keep="first" preserves already-filled action values from prior review sessions
        combined.drop_duplicates(subset=["new_name", "source"], keep="first", inplace=True)
        n_added = len(combined) - len(existing)
        combined.to_csv(rp, index=False)
        print("\n" + "=" * 60)
        print(f"Review file updated: +{max(n_added, 0)} new rows ({len(combined)} total) -> {rp}")
    else:
        new_review.to_csv(rp, index=False)
        print("\n" + "=" * 60)
        print(f"Review file: {rp}  ({len(new_review)} rows)")
    print("Fill \"action\" column (\"match\" or \"new\"), run 09_apply_fuzzy_review, then re-run.")

print("\n" + "=" * 60)
if _failed:
    print(f"Failed: {_failed}")
else:
    print("All WalterFootball sources completed.")

## 6 -- Validation

In [7]:
# Mini-validation: rows written by this source
import pandas as pd
from pathlib import Path
fr = etl.read_table("fact_rookie_rankings")
src_keys = list(SOURCES.keys())
sub = fr[fr["source_name"].isin(src_keys)]
print(f"Rows for this source: {len(sub)}")
print(sub.groupby(["source_name", "phase"]).size().to_string())
print()
print(f"fact_rookie_rankings total: {len(fr)} rows")
dupes = fr.duplicated(subset=["player_key","source_name","phase","draft_year"]).sum()
print(f"Composite key duplicates: {dupes} (expected 0)")

Rows for this source: 209
source_name        phase     
WalterFootball_DB  post_draft    42
WalterFootball_DL  post_draft    46
WalterFootball_LB  post_draft    28
WalterFootball_QB  post_draft    17
WalterFootball_RB  post_draft    16
WalterFootball_TE  post_draft    20
WalterFootball_WR  post_draft    40

fact_rookie_rankings total: 572 rows
Composite key duplicates: 0 (expected 0)
